# MiniMax-H3 × Google Colab（画像 → 動画・音つき）

上から順に、各セルの左の ▶ を押すだけ。終わると **OK** とだけ出ます。
問題があるときだけ、原因と直し方が出ます。

**始める前に必ず1回だけやること**
1. 上のメニュー **ランタイム → ランタイムのタイプを変更**
2. **GPU** を選ぶ → **A100**（無ければ L4）
3. **ハイメモリ（High-RAM）** を **オン**
4. 保存

**注意（お金の話）**
- これは **Colab Pro（月11.99ドル）以上**が必要です。**Google AI Pro には Colab Pro は含まれません**（別契約）。
- 無料のT4（16GB）では動きません。


## STEP 1 / GPUとメモリの確認

In [ ]:
#@title STEP 1: GPUとメモリの確認
import subprocess, shutil, sys

def fail(reason, fix):
    print("問題:", reason)
    print("直し方:", fix)
    raise SystemExit(1)

try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True).strip()
except Exception:
    fail("GPUが割り当てられていません。",
         "メニュー → ランタイム → ランタイムのタイプを変更 → GPU（A100推奨）→ 保存 → もう一度このセルを実行。")

GPU_NAME, mem = [x.strip() for x in out.split(",")]
VRAM_GB = int(mem.split()[0]) / 1024

import psutil
RAM_GB  = psutil.virtual_memory().total / 1024**3
DISK_GB = shutil.disk_usage("/content").free / 1024**3

print(f"GPU: {GPU_NAME} / VRAM {VRAM_GB:.0f}GB / RAM {RAM_GB:.0f}GB / 空きディスク {DISK_GB:.0f}GB")

if VRAM_GB < 21:
    fail(f"VRAMが {VRAM_GB:.0f}GB しかありません（MiniMax-H3 は 22GB以上が必要）。",
         "ランタイムのタイプを変更 → GPU を A100 に。A100が出ないときは L4 を選び直してください。")
if RAM_GB < 45:
    fail(f"本体メモリが {RAM_GB:.0f}GB しかありません（42GB分のモデルをここに逃がすため足りません）。",
         "ランタイムのタイプを変更 → ハイメモリ（High-RAM）をオン → 保存 → 再実行。")
if DISK_GB < 55:
    fail(f"空きディスクが {DISK_GB:.0f}GB しかありません（モデルに約43GB必要）。",
         "ランタイム → セッションの管理 → 不要なセッションを終了。または High-RAM ランタイムに変更。")

print("OK")


## STEP 2 / Googleドライブをつなぐ（できた動画の保存先）

In [ ]:
#@title STEP 2: Googleドライブをつなぐ
#@markdown モデル（約43GB）もドライブに残したいときだけ下にチェック。
#@markdown 残すと2回目以降のダウンロードは不要ですが、読み込みが遅くなります。
SAVE_MODELS_TO_DRIVE = False  #@param {type:"boolean"}

import os, shutil
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT  = "/content/drive/MyDrive/MiniMaxH3"
OUTPUT_DIR  = os.path.join(DRIVE_ROOT, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

if SAVE_MODELS_TO_DRIVE:
    MODEL_CACHE = os.path.join(DRIVE_ROOT, "models")
    os.makedirs(MODEL_CACHE, exist_ok=True)
    free = shutil.disk_usage(DRIVE_ROOT).free / 1024**3
    if free < 50:
        print("問題: Googleドライブの空きが", f"{free:.0f}GB", "しかありません（約43GB必要）。")
        print("直し方: 上のチェックを外して実行し直す（毎回ダウンロードになりますが動きます）。")
        raise SystemExit(1)
else:
    MODEL_CACHE = "/content/models_cache"
    os.makedirs(MODEL_CACHE, exist_ok=True)

print("動画の保存先:", OUTPUT_DIR)
print("OK")


## STEP 3 / ComfyUI（MiniMax-H3の実行エンジン）を入れる

In [ ]:
#@title STEP 3: 実行エンジンの導入（3〜5分）
import os, subprocess, sys

COMFY_DIR = "/content/ComfyUI"

def run(cmd, cwd=None):
    p = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str),
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if p.returncode != 0:
        print("問題: コマンドが失敗しました ->", cmd)
        print(p.stdout[-3000:])
        print("直し方: このセルをもう一度実行してください。それでも直らないときは")
        print("        ランタイム → ランタイムを接続解除して削除 → STEP1からやり直し。")
        raise SystemExit(1)
    return p.stdout

if not os.path.isdir(COMFY_DIR):
    run(["git", "clone", "--depth", "1",
         "https://github.com/comfyanonymous/ComfyUI.git", COMFY_DIR])

run([sys.executable, "-m", "pip", "install", "-q", "-r",
     os.path.join(COMFY_DIR, "requirements.txt")])
run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub[hf_transfer]"])

for sub in ("diffusion_models", "text_encoders", "vae"):
    os.makedirs(os.path.join(COMFY_DIR, "models", sub), exist_ok=True)
os.makedirs(os.path.join(COMFY_DIR, "input"), exist_ok=True)

print("OK")


## STEP 4 / MiniMax-H3 のモデルを取ってくる（約43GB・10〜25分）

回線が切れても、もう一度このセルを実行すれば続きから再開します。

In [ ]:
#@title STEP 4: モデルのダウンロード（約43GB）
import os, sys, subprocess

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import hf_hub_download

REPO = "Comfy-Org/MiniMax-H3"
FILES = [
    ("diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", "diffusion_models"),
    ("text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",       "text_encoders"),
    ("vae/minimax_h3_video_vae_fp16.safetensors",                         "vae"),
    ("vae/minimax_h3_audio_vae_fp32.safetensors",                         "vae"),
]

for remote, folder in FILES:
    name = os.path.basename(remote)
    dest = os.path.join(COMFY_DIR, "models", folder, name)
    if os.path.exists(dest) or os.path.islink(dest):
        print("済:", name); continue
    print("取得中:", name, "...")
    try:
        path = hf_hub_download(repo_id=REPO, filename=remote,
                               local_dir=MODEL_CACHE)
    except Exception as e:
        print("問題: ダウンロードに失敗しました ->", name)
        print("      ", type(e).__name__, str(e)[:300])
        print("直し方: このセルをもう一度実行（途中から再開します）。")
        print("        繰り返し失敗するときは https://huggingface.co/Comfy-Org/MiniMax-H3 が開けるか確認。")
        raise SystemExit(1)
    os.symlink(os.path.abspath(path), dest)
    print("完了:", name)

print("OK")


## STEP 5 / もとになる画像を1枚アップロード

In [ ]:
#@title STEP 5: 画像をアップロード
import os, shutil
from google.colab import files
from PIL import Image

up = files.upload()
if not up:
    print("問題: 画像が選ばれていません。")
    print("直し方: このセルをもう一度実行し、jpg か png を1枚選んでください。")
    raise SystemExit(1)

src_name = list(up.keys())[0]
INPUT_NAME = "input_frame" + os.path.splitext(src_name)[1].lower()
INPUT_PATH = os.path.join(COMFY_DIR, "input", INPUT_NAME)
shutil.move(src_name, INPUT_PATH)

with Image.open(INPUT_PATH) as im:
    SRC_W, SRC_H = im.size
print(f"画像: {INPUT_NAME} ({SRC_W}x{SRC_H})")
print("OK")


## STEP 6 / どんな動画にするかを決める

`PROMPT` に、動きと音を日本語でも英語でもよいので書きます。
`DURATION_SEC` は 2〜4 秒から始めるのが安全です（長いほど重い）。

In [ ]:
#@title STEP 6: 動画の内容を決める
PROMPT = "The subject in the image slowly comes to life. The camera makes a gentle, slow push-in. Soft natural light. Audio: quiet room tone with a soft ambient pad."  #@param {type:"string"}
DURATION_SEC = 2  #@param {type:"slider", min:1, max:6, step:1}
MEGAPIXELS = 0.4  #@param {type:"slider", min:0.2, max:1.0, step:0.1}
SEED = 0  #@param {type:"integer"}

import math, random

def pick_size(sw, sh, megapixels, multiple=32, max_short=768, max_long=1344):
    """H3の画布に合わせる: 短辺768まで・長辺1344まで・32の倍数。"""
    ar = sw / sh
    h = math.sqrt(megapixels * 1_000_000 / ar)
    w = h * ar
    scale = min(1.0, max_short / min(w, h), max_long / max(w, h))
    w, h = w * scale, h * scale
    w = max(multiple, int(round(w / multiple)) * multiple)
    h = max(multiple, int(round(h / multiple)) * multiple)
    return int(w), int(h)

def pick_length(seconds, fps=24, block=17, offset=5):
    """H3が受け付けるフレーム数 (17k+5) に切り上げる。"""
    n = max(offset, round(seconds * fps))
    return n + (offset - (n % block)) % block

WIDTH, HEIGHT = pick_size(SRC_W, SRC_H, MEGAPIXELS)
LENGTH = pick_length(DURATION_SEC)
NOISE_SEED = SEED if SEED else random.randint(1, 2**31 - 1)

print(f"サイズ: {WIDTH}x{HEIGHT} / フレーム数: {LENGTH}（約{LENGTH/24:.1f}秒・24fps）/ seed: {NOISE_SEED}")
print("OK")


## STEP 7 / 動画をつくる

最初の1回はモデルの読み込みに 5〜10 分かかります。そのあと生成が進みます。

In [ ]:
#@title STEP 7: 生成を実行
import json, os, subprocess, sys, time, urllib.request, urllib.error, uuid

PORT = 8188
BASE = f"http://127.0.0.1:{PORT}"

def api(path, payload=None, timeout=60):
    url = BASE + path
    if payload is None:
        req = urllib.request.Request(url)
    else:
        req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                     headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode())

# --- サーバー起動 -------------------------------------------------------
server = globals().get("_comfy_server")
if server is None or server.poll() is not None:
    log = open("/content/comfyui.log", "w")
    server = subprocess.Popen(
        [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", str(PORT),
         "--disable-auto-launch", "--disable-metadata"],
        cwd=COMFY_DIR, stdout=log, stderr=subprocess.STDOUT)
    globals()["_comfy_server"] = server

print("エンジンを起動しています ...")
object_info = None
for _ in range(180):
    if server.poll() is not None:
        print("問題: エンジンが起動できませんでした。")
        print(open("/content/comfyui.log").read()[-3000:])
        print("直し方: STEP3をもう一度実行してから、このセルを実行してください。")
        raise SystemExit(1)
    try:
        object_info = api("/object_info", timeout=10)
        break
    except Exception:
        time.sleep(2)
if object_info is None:
    print("問題: エンジンの応答がありません。")
    print(open("/content/comfyui.log").read()[-3000:])
    raise SystemExit(1)

if "MiniMaxH3ImageToVideo" not in object_info:
    print("問題: このComfyUIには MiniMax-H3 のノードが入っていません。")
    print("直し方: ランタイム → ランタイムを接続解除して削除 → STEP1からやり直し")
    print("        （最新のComfyUIを取り直します）。")
    raise SystemExit(1)

# --- ワークフロー（公式テンプレート video_minimax_h3_i2v と同じ構成）-----
G = {
    "1":  ["LoadImage",              {"image": INPUT_NAME}],
    "2":  ["UNETLoader",             {"unet_name": "minimax_h3_fl2va_pruned_int8_convrot.safetensors",
                                      "weight_dtype": "default"}],
    "3":  ["CLIPLoader",             {"clip_name": "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
                                      "type": "minimax", "device": "default"}],
    "4":  ["VAELoader",              {"vae_name": "minimax_h3_video_vae_fp16.safetensors"}],
    "5":  ["VAELoader",              {"vae_name": "minimax_h3_audio_vae_fp32.safetensors"}],
    "6":  ["MiniMaxH3ImageToVideo",  {"clip": ["3", 0], "vae": ["4", 0], "first_frame": ["1", 0],
                                      "prompt": PROMPT, "width": WIDTH, "height": HEIGHT,
                                      "length": LENGTH}],
    "7":  ["BasicGuider",            {"model": ["2", 0], "conditioning": ["6", 0]}],
    "8":  ["BasicScheduler",         {"model": ["2", 0], "scheduler": "simple",
                                      "steps": 20, "denoise": 1.0}],
    "9":  ["KSamplerSelect",         {"sampler_name": "res_multistep"}],
    "10": ["RandomNoise",            {"noise_seed": NOISE_SEED}],
    "11": ["SamplerCustomAdvanced",  {"noise": ["10", 0], "guider": ["7", 0],
                                      "sampler": ["9", 0], "sigmas": ["8", 0],
                                      "latent_image": ["6", 1]}],
    "12": ["VAEDecode",              {"samples": ["11", 0], "vae": ["4", 0]}],
    "13": ["VAEDecodeAudio",         {"samples": ["11", 0], "vae": ["5", 0]}],
    "14": ["CreateVideo",            {"images": ["12", 0], "audio": ["13", 0], "fps": 24.0}],
    "15": ["SaveVideo",              {"video": ["14", 0], "filename_prefix": "MiniMax_H3",
                                      "format": "auto", "codec": "auto"}],
}

def build(graph, info):
    """足りない必須項目は、このComfyUIの既定値で自動的に埋める。"""
    prompt = {}
    for nid, (cls, inputs) in graph.items():
        if cls not in info:
            print(f"問題: ノード {cls} がこのComfyUIにありません。")
            print("直し方: ランタイムを削除してSTEP1からやり直してください。")
            raise SystemExit(1)
        spec = info[cls]["input"].get("required", {})
        filled = dict(inputs)
        for key, val in spec.items():
            if key in filled:
                continue
            opts = val[1] if len(val) > 1 and isinstance(val[1], dict) else {}
            if "default" in opts:
                filled[key] = opts["default"]
            elif isinstance(val[0], list) and val[0]:
                filled[key] = val[0][0]
        filled = {k: v for k, v in filled.items() if k in spec or k in info[cls]["input"].get("optional", {})}
        prompt[nid] = {"class_type": cls, "inputs": filled}
    return prompt

prompt = build(G, object_info)
client_id = str(uuid.uuid4())

try:
    res = api("/prompt", {"prompt": prompt, "client_id": client_id})
except urllib.error.HTTPError as e:
    print("問題: ワークフローが受け付けられませんでした。")
    print(e.read().decode()[:3000])
    print("直し方: STEP6の秒数と画質を下げて、もう一度お試しください。")
    raise SystemExit(1)

pid = res["prompt_id"]
print("生成中です。初回はモデル読み込みで5〜10分かかります ...")

VIDEO_FILE = None
start = time.time()
while True:
    time.sleep(5)
    if server.poll() is not None:
        print("問題: 生成の途中でエンジンが落ちました（メモリ不足の可能性が高いです）。")
        print(open("/content/comfyui.log").read()[-3000:])
        print("直し方: STEP6で 秒数を2秒・画質を0.2 に下げて、STEP7をやり直してください。")
        raise SystemExit(1)
    hist = api(f"/history/{pid}")
    if pid in hist:
        entry = hist[pid]
        status = entry.get("status", {})
        if status.get("status_str") == "error":
            print("問題: 生成に失敗しました。")
            for m in status.get("messages", [])[-5:]:
                print(" ", m)
            print(open("/content/comfyui.log").read()[-2000:])
            print("直し方: STEP6で 秒数を2秒・画質を0.2 に下げて、STEP7をやり直してください。")
            raise SystemExit(1)
        for out in entry.get("outputs", {}).values():
            for item in (out.get("images") or []) + (out.get("videos") or []) + (out.get("gifs") or []):
                if str(item.get("filename", "")).lower().endswith((".mp4", ".webm", ".mkv")):
                    VIDEO_FILE = os.path.join(COMFY_DIR, "output",
                                              item.get("subfolder", ""), item["filename"])
        break
    if time.time() - start > 3600:
        print("問題: 1時間たっても終わりませんでした。")
        print("直し方: STEP6で 秒数を2秒・画質を0.2 に下げて、やり直してください。")
        raise SystemExit(1)

if not VIDEO_FILE or not os.path.exists(VIDEO_FILE):
    print("問題: 動画ファイルが見つかりません。")
    print(open("/content/comfyui.log").read()[-2000:])
    raise SystemExit(1)

print(f"かかった時間: {(time.time()-start)/60:.1f}分")
print("OK")


## STEP 8 / できた動画を見る＋Googleドライブへ保存

In [ ]:
#@title STEP 8: 動画の確認と保存
import os, shutil, base64, time
from IPython.display import HTML, display

saved = os.path.join(OUTPUT_DIR, f"MiniMaxH3_{time.strftime('%Y%m%d_%H%M%S')}.mp4")
shutil.copy(VIDEO_FILE, saved)

data = base64.b64encode(open(VIDEO_FILE, "rb").read()).decode()
display(HTML(f'<video width="640" controls src="data:video/mp4;base64,{data}"></video>'))

print("保存先:", saved)
print("OK")


---

## （任意）ComfyUI の画面を開く

ノートブックだけで完結させたい人は、ここは実行しなくて構いません。
実行すると URL が出ます。開くと、ノードをつないで細かく調整できる画面になります。
**Workflow → Browse Templates → Video → MiniMax H3** に公式のひな型が入っています。

In [ ]:
#@title （任意）ComfyUIの画面を開く
import subprocess, re, time, sys, os

subprocess.run("wget -q -O /content/cloudflared "
               "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
               "&& chmod +x /content/cloudflared", shell=True, check=True)

if globals().get("_comfy_server") is None or _comfy_server.poll() is not None:
    print("問題: エンジンが動いていません。")
    print("直し方: 先に STEP 7 を実行してください。")
    raise SystemExit(1)

tun = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(120):
    line = tun.stdout.readline()
    if not line:
        time.sleep(0.5); continue
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0); break
if not url:
    print("問題: 公開URLが取れませんでした。")
    print("直し方: このセルをもう一度実行してください。")
    raise SystemExit(1)

print("この URL を開いてください:", url)
print("OK")


---

**この環境は今後ComfyUIへ発展可能です**